In [ ]:
!pip install jiwer sacrebleu

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(0)

In [2]:
df = pd.read_excel("/kaggle/input/datasets/lordmostafa199/arabizi-arabic-parallel-corpora123/Arabizi-Arabic Parallel corpora.xlsx")
df.head()

,Arabize,Arabic,checked_word
0,so2al,سؤال,NaN
1,masa2,مساء,NaN
2,2sma2,أسماء,NaN
3,3marh,عمارة,1
4,s3adh,سعادة,1


## Fixing the Dataset

In [3]:
# Replace specific numbers with letters in the text column
df['Arabize'] = (
    df['Arabize']
    .astype(str)
    .str.replace('8', 'k', regex=False)
    .str.replace('6', 't', regex=False)
    .str.replace('9', 's', regex=False)
)
df.head(10)

,Arabize,Arabic,checked_word
0,so2al,سؤال,NaN
1,masa2,مساء,NaN
2,2sma2,أسماء,NaN
3,3marh,عمارة,1
4,s3adh,سعادة,1
5,ebda3,ابداع,NaN
6,mosha3′eb,مشاغب,NaN
7,msh3′ool,مشغول,NaN
8,hatha,هذا,NaN
9,thhb,ذهب,1


In [4]:
print(df.shape)
df.info()

(21499, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21499 entries, 0 to 21498
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Arabize       21499 non-null  object
 1   Arabic        21499 non-null  object
 2   checked_word  3996 non-null   object
dtypes: object(3)
memory usage: 504.0+ KB


In [5]:
# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# 1) Load the 20,000 rows
sub = df.iloc[:20000]
pairs = list(zip(sub["Arabize"], sub["Arabic"]))

PAD, SOS, EOS = 0, 1, 2


# Vocabulary mapping characters to integers
def make_vocab(strs):
  return {c: i + 3 for i, c in enumerate(sorted(set("".join(strs))))}


src_v = make_vocab([a for a, _ in pairs])
tgt_v = make_vocab([b for _, b in pairs])
inv_tgt = {i: c for c, i in tgt_v.items()}


# Create a Custom PyTorch Dataset
class ArabiziDataset(Dataset):

  def __init__(self, pairs, src_vocab, tgt_vocab):
    self.pairs = pairs
    self.src_vocab = src_vocab
    self.tgt_vocab = tgt_vocab

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    src_str, tgt_str = self.pairs[idx]
    src_ids = [self.src_vocab[c] for c in src_str if c in self.src_vocab]
    tgt_ids = (
        [SOS] + [self.tgt_vocab[c] for c in tgt_str if c in self.tgt_vocab] + [EOS]
    )
    return torch.tensor(src_ids, dtype=torch.long), torch.tensor(
        tgt_ids, dtype=torch.long
    )


# Collate function to handle dynamic padding per batch
def collate_fn(batch):
  srcs, tgts = zip(*batch)

  # Pad source sequences
  max_src_len = max(len(s) for s in srcs)
  padded_srcs = torch.full((len(srcs), max_src_len), PAD, dtype=torch.long)
  for i, s in enumerate(srcs):
    padded_srcs[i, : len(s)] = s

  # Pad target sequences
  max_tgt_len = max(len(t) for t in tgts)
  padded_tgts = torch.full((len(tgts), max_tgt_len), PAD, dtype=torch.long)
  for i, t in enumerate(tgts):
    padded_tgts[i, : len(t)] = t

  return padded_srcs, padded_tgts


# Initialize DataLoader with a safe batch size (e.g., 64)
dataset = ArabiziDataset(pairs, src_v, tgt_v)
dataloader = DataLoader(
    dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)


# 2) Model Definition
class Seq2Seq(nn.Module):

  def __init__(s, nsrc, ntgt, emb=64, hid=256, layers=4):
    super().__init__()
    s.se = nn.Embedding(nsrc, emb, padding_idx=PAD)
    s.te = nn.Embedding(ntgt, emb, padding_idx=PAD)
    s.enc = nn.LSTM(emb, hid, layers, batch_first=True)
    s.dec = nn.LSTM(emb, hid, layers, batch_first=True)
    s.out = nn.Linear(hid, ntgt)

  def forward(s, x, y_in):
    _, st = s.enc(s.se(x))
    o, _ = s.dec(s.te(y_in), st)
    return s.out(o)


V = len(tgt_v) + 3
m = Seq2Seq(len(src_v) + 3, V).to(device)

opt = torch.optim.Adam(m.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss(ignore_index=PAD)

# 3) Mini-batch Training Loop
epochs = 10  # With 20k rows, 10-20 epochs are usually plenty for a baseline
for ep in range(1, epochs + 1):
  m.train()
  total_loss = 0

  for X_batch, Y_batch in dataloader:
    X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

    logits = m(X_batch, Y_batch[:, :-1])
    loss = lossf(logits.reshape(-1, V), Y_batch[:, 1:].reshape(-1))

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(dataloader)
  print(f"Epoch {ep}/{epochs}, Loss: {round(avg_loss, 4)}")

Epoch 1/10, Loss: 2.4923
Epoch 2/10, Loss: 1.8152
Epoch 3/10, Loss: 1.0847
Epoch 4/10, Loss: 0.6226
Epoch 5/10, Loss: 0.4311
Epoch 6/10, Loss: 0.3317
Epoch 7/10, Loss: 0.2727
Epoch 8/10, Loss: 0.231
Epoch 9/10, Loss: 0.1972
Epoch 10/10, Loss: 0.1731


In [7]:
# Extract the test pairs from the remaining dataset
test_sub = df.iloc[20000:]
test_pairs = list(zip(test_sub["Arabize"], test_sub["Arabic"]))


@torch.no_grad()
def evaluate_test_set(pairs):
  m.eval()
  correct = 0
  total = len(pairs)

  for src_text, target_text in pairs:
    # Convert source text to tensor
    src_ids = [src_v[c] for c in src_text if c in src_v]
    if not src_ids:
      continue
    x = torch.tensor([src_ids], dtype=torch.long, device=device)

    # Greedy decoding for a single sample
    _, st = m.enc(m.se(x))
    tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
    outs = []

    for _ in range(len(src_text) * 2):  # Dynamic max length based on input size
      o, st = m.dec(m.te(tok), st)
      tok = m.out(o).argmax(-1)
      if tok.item() == EOS:
        break
      outs.append(tok.item())

    # Convert predicted IDs back to string
    pred_str = "".join([inv_tgt[i] for i in outs if i in inv_tgt])

    if pred_str == target_text:
      correct += 1

  accuracy = correct / total if total > 0 else 0
  print(f"Test Exact-Match Accuracy: {accuracy * 100:.2f}%")


# Run the evaluation
evaluate_test_set(test_pairs)

Test Exact-Match Accuracy: 51.17%


In [8]:
@torch.no_grad()
def translate_sentence_word_by_word(sentence):
    m.eval()

    # 1. Split the sentence into words using spaces
    words = sentence.split(" ")
    translated_words = []

    for word in words:
        # If the word is empty, preserve the space
        if not word.strip():
            translated_words.append(" ")
            continue

        # 2. Convert the word's characters into model input IDs
        src_ids = [src_v[c] for c in word if c in src_v]

        if not src_ids:
            # If no characters are found in the vocabulary, keep the original word
            translated_words.append(word)
            continue

        x = torch.tensor(
            [src_ids],
            dtype=torch.long,
            device=device
        )

        # 3. Encode the word and initialize the decoder
        _, st = m.enc(m.se(x))
        tok = torch.full(
            (1, 1),
            SOS,
            dtype=torch.long,
            device=device
        )
        decoded_ids = []

        # Generate the Arabic characters one by one
        for _ in range(len(word) * 2):
            o, st = m.dec(m.te(tok), st)
            tok = m.out(o).argmax(-1)

            predicted_id = tok.item()

            # Stop generating when the end-of-sequence token is predicted
            if predicted_id == EOS:
                break

            decoded_ids.append(predicted_id)

        # 4. Convert the predicted IDs back into characters
        pred_word = "".join([
            inv_tgt[i] for i in decoded_ids if i in inv_tgt
        ])

        translated_words.append(pred_word)

    # 5. Join the translated words with spaces
    return " ".join(translated_words)

In [9]:
@torch.no_grad()
def translate_arabizi_sentence(sentence):
  m.eval()

  # Map characters to source vocabulary IDs (ignore characters the model hasn't seen)
  src_ids = [src_v[c] for c in sentence if c in src_v]
  if not src_ids:
    return "Error: Input contains characters unknown to the model vocabulary."

  # Convert to tensor and send to device
  x = torch.tensor([src_ids], dtype=torch.long, device=device)

  # Encode
  _, st = m.enc(m.se(x))

  # Decode step-by-step using greedy search
  tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
  decoded_ids = []

  # Allow up to 2x the input length to complete the translation
  max_length = len(sentence) * 2
  for _ in range(max_length):
    o, st = m.dec(m.te(tok), st)
    tok = m.out(o).argmax(-1)

    predicted_id = tok.item()
    if predicted_id == EOS:
      break

    decoded_ids.append(predicted_id)

  # Convert predicted token IDs back to an Arabic string
  translated_sentence = "".join([inv_tgt[i] for i in decoded_ids if i in inv_tgt])
  return translated_sentence

# Test your inference engine with a sentence!
sample_input = "salam 3amel eh"
print(f"Input Arabizi: {sample_input}")
print(f"Translated Arabic: {translate_sentence_word_by_word(sample_input)}")


Input Arabizi: salam 3amel eh
Translated Arabic: سلام عامل هي


In [10]:
print(f"{'Input (Arabizi)':<20} | {'Target (Arabic)':<20} | {'Prediction':<20}")
print("-" * 65)

# Take the first 20 pairs from your test set
for src_text, target_text in test_pairs[:20]:
  pred_str = translate_arabizi_sentence(src_text)
  print(f"{src_text:<20} | {target_text:<20} | {pred_str:<20}")

Input (Arabizi)      | Target (Arabic)      | Prediction          
-----------------------------------------------------------------
btlt                 | بطلت                 | بطل                 
b3'yrha              | بغيرها               | بغيرها              
flstenyeh            | فلسطينيه             | فلسطيني             
$ele                 | شيلي                 | شيلي                
yjakr                | يجاكر                | يجاكر               
bjamtk               | بجامتك               | بجامتك              
Norti                | نورتي                | نرتي                
nosi                 | نصي                  | نصي                 
elthani              | الثاني               | الثاني              
bshofo               | بشوفو                | بشوفو               
7babeh               | حبابه                | حبابه               
tybeh                | طيبه                 | طيبه                
sborah               | صبوره                | سبوره            

In [12]:
from jiwer import cer, wer
import sacrebleu

@torch.no_grad()
def evaluate_advanced_with_bleu(pairs):
    m.eval()
    exact_matches = 0
    total = len(pairs)
    
    preds = []
    targets = []

    for src_text, target_text in pairs:
        src_ids = [src_v[c] for c in src_text if c in src_v]
        if not src_ids:
            continue
        x = torch.tensor([src_ids], dtype=torch.long, device=device)

        _, st = m.enc(m.se(x))
        tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
        outs = []

        for _ in range(len(src_text) * 2):
            o, st = m.dec(m.te(tok), st)
            tok = m.out(o).argmax(-1)
            if tok.item() == EOS:
                break
            outs.append(tok.item())

        pred_str = "".join([inv_tgt[i] for i in outs if i in inv_tgt])
        
        if pred_str == target_text:
            exact_matches += 1
            
        preds.append(pred_str)
        targets.append(target_text)

    exact_match_acc = (exact_matches / total) * 100
    character_error_rate = cer(targets, preds) * 100
    
    # Calculate BLEU score (sacrebleu expects a list of predictions and a list of lists for references)
    bleu_result = sacrebleu.corpus_bleu(preds, [targets])

    print(f"--- Comprehensive Evaluation Results ---")
    print(f"Exact-Match Accuracy      : {exact_match_acc:.2f}%")
    print(f"Character Error Rate (CER): {character_error_rate:.2f}% (Lower is better)")
    print(f"BLEU Score                : {bleu_result.score:.2f} (Higher is better)")

# Run it on your test set after installing the packages
evaluate_advanced_with_bleu(test_pairs)

--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 51.17%
Character Error Rate (CER): 15.04% (Lower is better)
BLEU Score                : 0.00 (Higher is better)


Character-level translation models generate text character-by-character rather than word-by-word, making them uniquely resilient to spelling errors, inflection changes, and morphologically rich languages (like Arabic, Finnish, or Turkish), where standard word-based metrics often fail

Evaluating their performance requires a combination of character-specific automated metrics, neural semantic scores, and human evaluation.

Edits: Insertion, Deletion, Change
* Levenshtein Distance: number of Edits in charachter per word
* Character Error Rate (CER): Levenshtein Distance/len(word) (it is normalized)
* Word Error Rate (WER): number of Edits in Words/len(sentence)

In [13]:
# 1. Extract the training pairs used to train the model (first 20,000 rows)
train_sub = df.iloc[:20000]
train_pairs = list(zip(train_sub["Arabize"], train_sub["Arabic"]))

# 2. Run your advanced evaluation function on the training set
print("Evaluating on Training Data:")
evaluate_advanced_with_bleu(train_pairs)

Evaluating on Training Data:
--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 65.31%
Character Error Rate (CER): 7.96% (Lower is better)
BLEU Score                : 0.00 (Higher is better)


In [15]:
from jiwer import cer, wer
import sacrebleu
from sacrebleu.metrics import CHRF, TER

In [18]:


@torch.no_grad()
def evaluate_advanced_comprehensive(pairs):
    m.eval()
    exact_matches = 0
    total = len(pairs)
    
    preds = []
    targets = []

    for src_text, target_text in pairs:
        src_ids = [src_v[c] for c in src_text if c in src_v]
        if not src_ids:
            continue
        x = torch.tensor([src_ids], dtype=torch.long, device=device)

        _, st = m.enc(m.se(x))
        tok = torch.full((1, 1), SOS, dtype=torch.long, device=device)
        outs = []

        for _ in range(len(src_text) * 2):
            o, st = m.dec(m.te(tok), st)
            tok = m.out(o).argmax(-1)
            if tok.item() == EOS:
                break
            outs.append(tok.item())

        pred_str = "".join([inv_tgt[i] for i in outs if i in inv_tgt])
        
        if pred_str == target_text:
            exact_matches += 1
            
        preds.append(pred_str)
        targets.append(target_text)

    # Core Metrics Calculations
    exact_match_acc = (exact_matches / total) * 100
    character_error_rate = cer(targets, preds) * 100
    
    # BLEU, chrF, and TER via sacrebleu
    bleu_result = sacrebleu.corpus_bleu(preds, [targets])
    
    chrf_metric = CHRF()
    chrf_result = chrf_metric.corpus_score(preds, [targets])
    
    ter_metric = TER()
    ter_result = ter_metric.corpus_score(preds, [targets])

    # Print Report
    print(f"--- Comprehensive Evaluation Results ---")
    print(f"Exact-Match Accuracy      : {exact_match_acc:.2f}% (Higher is better)")
    print(f"Character Error Rate (CER): {character_error_rate:.2f}% (Lower is better)")
    print(f"chrF Score                : {chrf_result.score:.2f} (Higher is better)")
    print(f"TER Score                 : {ter_result.score:.2f}% (Lower is better)")

# 1. Run evaluation on your Test Set
print("=== Evaluating on Test Data ===")
evaluate_advanced_comprehensive(test_pairs)

# 2. Extract and run evaluation on your Training Set
train_sub = df.iloc[:20000]
train_pairs = list(zip(train_sub["Arabize"], train_sub["Arabic"]))

print("\n=== Evaluating on Training Data ===")
evaluate_advanced_comprehensive(train_pairs)

=== Evaluating on Test Data ===
--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 51.17% (Higher is better)
Character Error Rate (CER): 15.04% (Lower is better)
chrF Score                : 65.93 (Higher is better)
TER Score                 : 48.10% (Lower is better)

=== Evaluating on Training Data ===
--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 65.31% (Higher is better)
Character Error Rate (CER): 7.96% (Lower is better)
chrF Score                : 82.46 (Higher is better)
TER Score                 : 28.97% (Lower is better)


In [20]:
# 1) Model Definition with Dropout enabled 
# (Dropout is required so R-Drop has different sub-models to regularize)
class Seq2SeqRDrop(nn.Module):
  def __init__(self, nsrc, ntgt, emb=64, hid=256, layers=4, dropout=0.2):
    super().__init__()
    self.se = nn.Embedding(nsrc, emb, padding_idx=PAD)
    self.te = nn.Embedding(ntgt, emb, padding_idx=PAD)
    self.enc = nn.LSTM(emb, hid, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
    self.dec = nn.LSTM(emb, hid, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
    self.out = nn.Linear(hid, ntgt)

  def forward(self, x, y_in):
    _, st = self.enc(self.se(x))
    o, _ = self.dec(self.te(y_in), st)
    return self.out(o)

V = len(tgt_v) + 3
# Initialize model with active dropout
m = Seq2SeqRDrop(len(src_v) + 3, V, dropout=0.2).to(device)

opt = torch.optim.Adam(m.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss(ignore_index=PAD)

# R-Drop hyperparameter alpha (paper sets alpha = 5 for machine translation)
alpha = 5.0

# 2) Mini-batch Training Loop with R-Drop
epochs = 10 
for ep in range(1, epochs + 1):
  m.train()
  total_loss = 0

  for X_batch, Y_batch in dataloader:
    X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
    batch_size = X_batch.size(0)

    # Efficiency Trick: Duplicate input data once [x; x] and concatenate 
    # along batch dimension to perform both forward passes in a single batch[cite: 7]
    X_repeated = torch.cat([X_batch, X_batch], dim=0)
    Y_repeated = torch.cat([Y_batch, Y_batch], dim=0)

    # Forward pass (dropout samples two different sub-models for each half)[cite: 6]
    logits_repeated = m(X_repeated, Y_repeated[:, :-1])

    # Split logits back into two separate distributions P1 and P2
    logits1 = logits_repeated[:batch_size]
    logits2 = logits_repeated[batch_size:]

    # 3) Calculate Negative Log-Likelihood (NLL) Loss for both sub-models[cite: 6]
    target_tokens = Y_batch[:, 1:]
    loss1 = lossf(logits1.reshape(-1, V), target_tokens.reshape(-1))
    loss2 = lossf(logits2.reshape(-1, V), target_tokens.reshape(-1))
    nll_loss = loss1 + loss2

    # 4) Calculate Bidirectional KL-Divergence Loss (masked for padding tokens)[cite: 6]
    log_p1 = F.log_softmax(logits1, dim=-1)
    log_p2 = F.log_softmax(logits2, dim=-1)
    p1 = F.softmax(logits1, dim=-1)
    p2 = F.softmax(logits2, dim=-1)

    kl1 = F.kl_div(log_p1, p2, reduction='none').sum(dim=-1)
    kl2 = F.kl_div(log_p2, p1, reduction='none').sum(dim=-1)

    # Mask out PAD tokens so they don't corrupt the KL divergence calculation
    pad_mask = (target_tokens == PAD)
    kl1 = kl1.masked_fill(pad_mask, 0.0)
    kl2 = kl2.masked_fill(pad_mask, 0.0)
    
    num_non_pad = (~pad_mask).sum()
    kl_loss = (kl1.sum() + kl2.sum()) / (2.0 * num_non_pad if num_non_pad > 0 else 1.0)

    # 5) Final R-Drop Objective: NLL Loss + alpha * KL Loss[cite: 6]
    loss = nll_loss + alpha * kl_loss

    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(dataloader)
  print(f"Epoch {ep}/{epochs}, R-Drop Loss: {round(avg_loss, 4)}")

Epoch 1/10, R-Drop Loss: 5.1739
Epoch 2/10, R-Drop Loss: 3.8468
Epoch 3/10, R-Drop Loss: 2.8192
Epoch 4/10, R-Drop Loss: 1.9461
Epoch 5/10, R-Drop Loss: 1.4032
Epoch 6/10, R-Drop Loss: 1.1135
Epoch 7/10, R-Drop Loss: 0.9288
Epoch 8/10, R-Drop Loss: 0.7996
Epoch 9/10, R-Drop Loss: 0.7159
Epoch 10/10, R-Drop Loss: 0.6454


In [21]:
# 1. Run evaluation on your Test Set
print("=== Evaluating on Test Data ===")
evaluate_advanced_comprehensive(test_pairs)

# 2. Extract and run evaluation on your Training Set
train_sub = df.iloc[:20000]
train_pairs = list(zip(train_sub["Arabize"], train_sub["Arabic"]))

print("\n=== Evaluating on Training Data ===")
evaluate_advanced_comprehensive(train_pairs)

=== Evaluating on Test Data ===
--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 51.03% (Higher is better)
Character Error Rate (CER): 14.11% (Lower is better)
chrF Score                : 65.46 (Higher is better)
TER Score                 : 48.10% (Lower is better)

=== Evaluating on Training Data ===
--- Comprehensive Evaluation Results ---
Exact-Match Accuracy      : 60.53% (Higher is better)
Character Error Rate (CER): 9.17% (Lower is better)
chrF Score                : 77.65 (Higher is better)
TER Score                 : 33.91% (Lower is better)
